# Ensemble Demo on a Real Dataset 
For this demo, tree-based ensembles often outperform a simple linear model on this dataset.

**Models compared**
- Logistic Regression
- Decision Tree
- Bagging + Decision Tree
- Random Forest
- AdaBoost + Decision Stump
- AdaBoost + Shallow Tree

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier


## 1. Load real data

In [11]:
# Dataset:
# U.S. Forest Service cartographic variables for forest cover type prediction.
# Official scikit-learn loader:
# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html

X, y = fetch_covtype(return_X_y=True, shuffle=True, random_state=42)

print("Original shape:", X.shape)
print("Number of classes:", len(np.unique(y)))


Original shape: (581012, 54)
Number of classes: 7


## 2. Subsample for faster classroom demo

In [12]:
# The full dataset is large. For a faster in-class demo, use a subset.
# You can increase n_demo later if you want.
n_demo = 20000

rng = np.random.RandomState(42)
idx = rng.choice(len(X), size=n_demo, replace=False)

X_demo = X[idx]
y_demo = y[idx]

print("Demo shape:", X_demo.shape)
print("Class counts:")
print(pd.Series(y_demo).value_counts().sort_index())


Demo shape: (20000, 54)
Class counts:
1     7089
2    10009
3     1192
4       99
5      320
6      588
7      703
Name: count, dtype: int64


## 3. Train / test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo,
    test_size=0.2,
    random_state=42,
    stratify=y_demo
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (16000, 54) Test: (4000, 54)


## 4. Define models

In [14]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1500,
          ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=None
    ),

    "Bagging + Decision Tree": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=80,
        random_state=42,
        n_jobs=-1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        random_state=42,
        n_jobs=-1
    ),

    "AdaBoost + Decision Stump": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=120,
        learning_rate=0.8,
        random_state=42
    ),

    "AdaBoost + Shallow Tree": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
        n_estimators=120,
        learning_rate=0.8,
        random_state=42
    ),
}


## 5. Evaluation helper

In [15]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test, cv_splits=3):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    test_recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    test_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    cv_result = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_weighted": "f1_weighted"
        },
        n_jobs=-1
    )

    return {
        "model": name,
        "test_accuracy": test_accuracy,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_f1": test_f1,
        "cv_accuracy_mean": np.mean(cv_result["test_accuracy"]),
        "cv_accuracy_std": np.std(cv_result["test_accuracy"]),
        "cv_f1_mean": np.mean(cv_result["test_f1_weighted"]),
    }


## 6. Run comparison

In [16]:
results = []

for name, model in models.items():
    print(f"Running: {name}")
    row = evaluate_model(name, model, X_train, y_train, X_test, y_test, cv_splits=3)
    results.append(row)

results_df = pd.DataFrame(results).sort_values(
    by=["test_accuracy", "cv_accuracy_mean"],
    ascending=False
).reset_index(drop=True)

results_df.style.format({
    "test_accuracy": "{:.4f}",
    "test_precision": "{:.4f}",
    "test_recall": "{:.4f}",
    "test_f1": "{:.4f}",
    "cv_accuracy_mean": "{:.4f}",
    "cv_accuracy_std": "{:.4f}",
    "cv_f1_mean": "{:.4f}",
})


Running: Logistic Regression
Running: Decision Tree
Running: Bagging + Decision Tree
Running: Random Forest
Running: AdaBoost + Decision Stump
Running: AdaBoost + Shallow Tree


,model,test_accuracy,test_precision,test_recall,test_f1,cv_accuracy_mean,cv_accuracy_std,cv_f1_mean
0,Bagging + Decision Tree,0.8417,0.8408,0.8417,0.8389,0.8155,0.0043,0.8109
1,Random Forest,0.8333,0.8328,0.8333,0.8277,0.8099,0.0036,0.8030
2,Decision Tree,0.7600,0.7613,0.7600,0.7605,0.7308,0.0047,0.7312
3,Logistic Regression,0.7242,0.7092,0.7242,0.7143,0.7248,0.0076,0.7144
4,AdaBoost + Decision Stump,0.6420,0.6002,0.6420,0.6171,0.6469,0.0027,0.6238
5,AdaBoost + Shallow Tree,0.6070,0.6116,0.6070,0.6024,0.6047,0.0116,0.6040


## 7. Focused view: base vs ensemble

In [17]:
focus = results_df[results_df["model"].isin([
    "Logistic Regression",
    "Decision Tree",
    "Bagging + Decision Tree",
    "Random Forest",
    "AdaBoost + Decision Stump",
    "AdaBoost + Shallow Tree",
])].copy()

focus.style.format({
    "test_accuracy": "{:.4f}",
    "test_precision": "{:.4f}",
    "test_recall": "{:.4f}",
    "test_f1": "{:.4f}",
    "cv_accuracy_mean": "{:.4f}",
    "cv_accuracy_std": "{:.4f}",
    "cv_f1_mean": "{:.4f}",
})


,model,test_accuracy,test_precision,test_recall,test_f1,cv_accuracy_mean,cv_accuracy_std,cv_f1_mean
0,Bagging + Decision Tree,0.8417,0.8408,0.8417,0.8389,0.8155,0.0043,0.8109
1,Random Forest,0.8333,0.8328,0.8333,0.8277,0.8099,0.0036,0.8030
2,Decision Tree,0.7600,0.7613,0.7600,0.7605,0.7308,0.0047,0.7312
3,Logistic Regression,0.7242,0.7092,0.7242,0.7143,0.7248,0.0076,0.7144
4,AdaBoost + Decision Stump,0.6420,0.6002,0.6420,0.6171,0.6469,0.0027,0.6238
5,AdaBoost + Shallow Tree,0.6070,0.6116,0.6070,0.6024,0.6047,0.0116,0.6040


## 8. Simple interpretation prompts

In [18]:
best_row = results_df.iloc[0]
print("Best model on this run:", best_row["model"])
print("Best test accuracy:", round(best_row["test_accuracy"], 4))
print()

print("Suggested discussion:")
print("- Does Random Forest beat a single Decision Tree?")
print("- Does bagging improve the high-variance tree?")
print("- Does AdaBoost with weak trees outperform the single weak tree idea?")
print("- Does the ensemble beat Logistic Regression on this real dataset?")


Best model on this run: Bagging + Decision Tree
Best test accuracy: 0.8418

Suggested discussion:
- Does Random Forest beat a single Decision Tree?
- Does bagging improve the high-variance tree?
- Does AdaBoost with weak trees outperform the single weak tree idea?
- Does the ensemble beat Logistic Regression on this real dataset?
